# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JawadKhan65/glamourrr/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [10]:
import pandas as pd
import numpy as np

# Loading the dataset
data_url = 'https://raw.githubusercontent.com/JawadKhan65/glamourrr/main/data/raw/content_refresh_anonymized.csv'
try:
    df = pd.read_csv(data_url)
    print("Dataset loaded successfully.")
except Exception as e:
    print(f"Error: {e}")

# Cleaning
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# Categorical
categorical_cols = [c for c in df.select_dtypes(exclude=[np.number]).columns if 'id' not in c.lower()]
df = pd.get_dummies(df, columns=categorical_cols, dummy_na=True)

# Define Feature Vector - Specifically excluding the target proxy 'avg_position'
exclude_patterns = ['id', 'label', 'target', 'avg_position', 'provider_used']
features = [col for col in df.columns if not any(pat in col.lower() for pat in exclude_patterns)]
X = df[features]

print(f"Clean feature vector built with {X.shape[1]} features (avg_position excluded).")
display(X.head())

Dataset loaded successfully.
Clean feature vector built with 85 features (avg_position excluded).


,search_volume,competition,cpc,word_count,char_count,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,...,position_tier_page_3_5,position_tier_striking,position_tier_top_3,position_tier_nan,trend_direction_down,trend_direction_flat,trend_direction_new,trend_direction_stable,trend_direction_up,trend_direction_nan
0,10.0,0.67,2.05,3221.0,20457.0,3803,29,22,17,16,...,False,True,False,False,True,False,False,False,False,False
1,90.0,0.01,0.05,2481.0,15562.0,15320,7,10,9,9,...,True,False,False,False,True,False,False,False,False,False
2,0.0,0.00,0.00,3515.0,23643.0,12581,11,14,11,11,...,True,False,False,False,True,False,False,False,False,False
3,10.0,0.00,0.00,2877.0,19116.0,11751,58,87,78,75,...,False,False,False,False,False,False,False,True,False,False
4,0.0,0.00,0.00,2803.0,17469.0,19140,24,177,145,144,...,True,False,False,False,True,False,False,False,False,False


## 2. Feature notes (meaning, missing, categorical, available-when?)

* **search_volume / competition / cpc**: External market signals. Missing values filled with median. Available at inference.
* **word_count / char_count**: Content length metrics. Used to determine content depth.
* **impressions_90d / clicks_90d**: Historical performance metrics. These are retrospective but available at the time of ranking.
* **Categorical Tiers**: One-hot encoded versions of `age_tier`, `freshness_tier`, etc., to capture non-linear relationships.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [7]:
# Using 'avg_position' as the focus for the leakage hunt since it is a primary FlyRank signal
target_col = 'avg_position'

print(f"Performing leakage check against: {target_col}")

# Ensure we only correlate against numeric features present in X
numeric_X = X.select_dtypes(include=[np.number])
correlation_series = numeric_X.corrwith(df[target_col]).sort_values(ascending=False)

print(f"\nTop 10 correlations with {target_col}:")
print(correlation_series.head(10))

# Flag potential leakage (> 0.95)
leakage_suspects = correlation_series[abs(correlation_series) > 0.95]
if not leakage_suspects.empty:
    print(f"\nWARNING: Potential leakage or redundant columns detected: {leakage_suspects.index.tolist()}")
else:
    print("\nNo high-correlation leakage detected against avg_position.")

Performing leakage check against: avg_position

Top 10 correlations with avg_position:
avg_position       1.000000
age_tier_order     0.151706
word_count         0.090661
competition        0.071908
char_count         0.071377
cpc                0.053804
search_volume      0.048418
trend_pct          0.045188
ai_sessions_90d    0.035971
ai_traffic_pct     0.006543
dtype: float64



## 4. What I excluded and why

* **content_id / client_id**: Unique identifiers that would cause overfitting.
* **provider_used**: Excluded as a safeguard against potential bias/leakage patterns.
* **avg_position**: Excluded because it is our primary performance target/proxy for this check; including it would lead to trivial leakage.

In [11]:
# Final verification of the clean feature set
print(f"Final feature set count: {len(features)}")
print(f"Feature matrix X shape: {X.shape}")

# Ensure target proxy is not in features
assert 'avg_position' not in X.columns, "Leakage Alert: avg_position still in X!"
print("\nSuccess: Target proxy 'avg_position' successfully removed from features.")

# Display summary of columns kept vs excluded
excluded = [col for col in df.columns if col not in X.columns]
print(f"\nTotal columns in processed dataframe: {len(df.columns)}")
print(f"Total columns excluded: {len(excluded)}")

Final feature set count: 85
Feature matrix X shape: (30000, 85)

Success: Target proxy 'avg_position' successfully removed from features.

Total columns in processed dataframe: 89
Total columns excluded: 4


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.